# Model Parameter Count Calculations

This notebook computes total and trainable parameter counts for the five model families used in Phases 2 and 3, using official backend implementations and default non-pretrained configuration.

The defaults are size-balanced to be around 30M parameters when feasible: AST (~31.7M), ConvNeXt (~27.8M), SSAMBA (~31.9M), xLSTM (~35.8M).
For `mlp_mixer`, the closest official timm variant currently used is `gmixer_24_224` (~24.1M).

In [1]:
import pandas as pd

from speech_recognition.config import ModelConfig
from speech_recognition.models.registry import build_model_adapter

In [2]:
families = ["ast", "convnext", "ssamba", "xlstm", "mlp_mixer"]
rows = []
for family in families:
    config = ModelConfig(family=family, num_classes=12, pretrained=False)
    adapter = build_model_adapter(family=family, model_config=config)
    total_params = sum(p.numel() for p in adapter.parameters())
    trainable_params = sum(p.numel() for p in adapter.parameters() if p.requires_grad)
    rows.append(
        {
            "model": family,
            "source_library": adapter.source_library,
            "backbone_class": adapter.backbone.__class__.__name__,
            "total_parameters": total_params,
            "trainable_parameters": trainable_params,
            "ast_hidden_size": config.ast_hidden_size if family == "ast" else None,
            "ast_num_hidden_layers": config.ast_num_hidden_layers if family == "ast" else None,
            "ast_num_attention_heads": config.ast_num_attention_heads if family == "ast" else None,
            "ast_intermediate_size": config.ast_intermediate_size if family == "ast" else None,
            "ssamba_d_model": config.ssamba_d_model if family == "ssamba" else None,
            "ssamba_d_state": config.ssamba_d_state if family == "ssamba" else None,
            "ssamba_num_layers": config.ssamba_num_layers if family == "ssamba" else None,
            "xlstm_dim": config.xlstm_dim if family == "xlstm" else None,
            "xlstm_num_blocks": config.xlstm_num_blocks if family == "xlstm" else None,
        }
    )

df = pd.DataFrame(rows)
df

Mamba SSM macOS: Running on Apple Silicon with MPS acceleration


,model,source_library,backbone_class,total_parameters,trainable_parameters,ast_hidden_size,ast_num_hidden_layers,ast_num_attention_heads,ast_intermediate_size,ssamba_d_model,ssamba_d_state,ssamba_num_layers,xlstm_dim,xlstm_num_blocks
0,ast,transformers,ASTBackboneWrapper,31720972,31720972,512.0,10.0,8.0,2048.0,NaN,NaN,NaN,NaN,NaN
1,convnext,torchvision,ConvNeXt,27826284,27826284,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ssamba,mamba-ssm,MambaHead,31947276,31947276,NaN,NaN,NaN,NaN,768.0,64.0,8.0,NaN,NaN
3,xlstm,xlstm,XLSTMHead,35781172,35781172,NaN,NaN,NaN,NaN,NaN,NaN,NaN,768.0,10.0
4,mlp_mixer,timm,MlpMixer,24144108,24144108,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
df.sort_values("model").to_csv("../outputs/model_parameter_counts.csv", index=False)
print("Saved ../outputs/model_parameter_counts.csv")

Saved ../outputs/model_parameter_counts.csv
